# 07. 떨어진 원문 구간을 합친 인용 수정

이 노트북을 기존 Phase 4 폴더의 `notebooks/`에 넣고 기존 커널에서 실행하세요. 앞서 06번 수정이 적용된 프로젝트를 대상으로 합니다.

이번 원인은 공백 차이가 아닙니다. 모델이 `<부가세 포함, 금액단위 : 원>` 뒤에 있던 발주자·하도급 관련 내용을 건너뛰고 표 본문을 이어 붙여 하나의 인용문으로 만들었습니다.
원문 문자 위치 0–18의 표제와 145–213의 표 본문은 연속 구간이 아닙니다. 이 두 구간을 **별도 인용 두 개로 표시**하고 각각 원문의 정확한 문자 위치와 대조합니다.

이 수정은 같은 청크에서 전체 줄 단위로 일치하는 두 구간의 배치가 하나로 확정되는 경우만 허용합니다. 숫자·단어·VAT·문장부호·순서를 바꾸지 않고, 줄 내부 단어를 생략하거나 다른 문서를 찾아 맞추지 않습니다.
중간 생략 구간과 원래 모델 인용문은 진단 기록에 남깁니다. 인용 출처 검증과 주장의 의미적 정답은 별개입니다.

**추가 API 호출 0회**로 기존 캐시의 모델 응답을 재검증합니다. 데이터·인덱스·캐시·원래 실패 결과는 보존하며 수정 코드는 백업합니다.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = None
FAILED_RESULT = 'phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_003.json'
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'rfp_phase4_local_20261111']
    ROOT = next((p.resolve() for p in candidates
                 if (p / 'rfp_phase4/runtime.py').is_file()
                 and (p / 'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None:
    raise FileNotFoundError('기존 Phase 4 폴더의 notebooks/에 저장하거나 PROJECT_ROOT를 지정하세요.')
print('Python:', sys.executable)
print('프로젝트:', ROOT)
print('기존 결과:', ROOT / FAILED_RESULT)
print('추가 API 호출: 0회')


Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
프로젝트: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111
기존 결과: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_003.json
추가 API 호출: 0회


## 1. v2 수정 적용

수정 파일이 내장되어 있습니다. 기존 파일 해시를 확인하고 백업한 뒤 적용합니다. 이미 적용했다면 건너뜁니다.
06번 수정이 없거나 사용자가 직접 바꾼 코드와 충돌하면 덮어쓰지 않고 멈춥니다.


In [2]:
INSTALLER_SOURCE = '"""Upgrade citation recovery v1 to whole-line split recovery v2, without API."""\nfrom datetime import datetime,timezone\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nPATCH_ID=\'phase4-citation-split-fix-v2\'\nALLOWED={\'rfp_phase4/citation_recovery.py\',\'tests/test_phase4_citation_split.py\'}\n\n\ndef sha(content):return hashlib.sha256(content).hexdigest()\n\n\ndef apply_patch(project,patch_root=None,*,manifest=None,embedded_files=None):\n    project=Path(project).resolve();patch_root=Path(patch_root or Path(__file__).resolve().parent)\n    if manifest is None:manifest=json.loads((patch_root/\'patch_manifest.json\').read_text(encoding=\'utf-8\'))\n    if manifest.get(\'patch_id\')!=PATCH_ID or set(manifest.get(\'files\',{}))!=ALLOWED:\n        raise ValueError(\'수정 파일 목록 또는 버전이 올바르지 않습니다.\')\n    if not (project/\'rfp_phase4/runtime.py\').is_file() or not (project/\'index/index_meta.json\').is_file():\n        raise ValueError(\'기존 Phase 4 프로젝트 최상위 폴더를 지정하세요.\')\n    staged={};skipped=[]\n    for rel,spec in manifest[\'files\'].items():\n        target=(project/rel).resolve();target.relative_to(project)\n        content=embedded_files[rel].encode(\'utf-8\') if embedded_files is not None else (patch_root/\'payload\'/rel).read_bytes()\n        if sha(content)!=spec[\'new_sha256\']:raise ValueError(\'수정 내용 해시 불일치: \'+rel)\n        old=target.read_bytes() if target.is_file() else None\n        digest=sha(old) if old is not None else None\n        if digest==spec[\'new_sha256\']:skipped.append(rel);continue\n        if digest!=spec[\'old_sha256\']:\n            raise ValueError(\'06번 수정이 적용되지 않았거나 직접 수정한 코드입니다: \'+rel)\n        staged[rel]=content\n    nbpath=project/\'notebooks/05_younggwan_phase4_grounded_answers.ipynb\'\n    if nbpath.is_file():\n        nb=json.loads(nbpath.read_text(encoding=\'utf-8\'));changed=False\n        for cell in nb.get(\'cells\',[]):\n            text=\'\'.join(cell[\'source\']) if isinstance(cell.get(\'source\'),list) else cell.get(\'source\',\'\')\n            if cell.get(\'cell_type\')==\'code\' and \'def show_result(result):\' in text:\n                updated=text.replace(\'공백 차이를 원문으로 복원한 인용 수:\', \'복원·분리한 인용 수:\')\n                if updated!=text:cell[\'source\']=updated;changed=True\n        if changed:staged[str(nbpath.relative_to(project))]=(json.dumps(nb,ensure_ascii=False,indent=2)+\'\\n\').encode(\'utf-8\')\n    backup=None;written=[]\n    try:\n        if staged:\n            backup=project/\'phase4_outputs/patch_backups\'/datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n            backup.mkdir(parents=True)\n            for rel,content in staged.items():\n                target=project/rel\n                if target.is_file():\n                    old=backup/rel;old.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(target,old)\n                target.parent.mkdir(parents=True,exist_ok=True)\n                temp=target.with_name(target.name+\'.split-patch-tmp\');temp.write_bytes(content);temp.replace(target);written.append(rel)\n    except Exception:\n        for rel in reversed(written):\n            old=backup/rel\n            if old.is_file():shutil.copy2(old,project/rel)\n            else:(project/rel).unlink(missing_ok=True)\n        raise\n    result={\'patch_id\':PATCH_ID,\'modified_files\':list(staged),\'already_applied_files\':skipped,\n            \'backup_directory\':str(backup) if backup else None,\'api_calls\':0,\n            \'data_or_index_modified\':False,\'cache_modified\':False,\'restart_existing_phase4_notebook_kernel\':True}\n    if backup:(backup/\'patch_report.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return result\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--project\',type=Path,default=Path.cwd())\n    print(json.dumps(apply_patch(parser.parse_args().project),ensure_ascii=False,indent=2))\n'
PATCH_MANIFEST = {'patch_id': 'phase4-citation-split-fix-v2', 'requires': 'phase4-citation-fix-v1', 'files': {'rfp_phase4/citation_recovery.py': {'old_sha256': '5705ba107252be370b0598fdc603a628d7f6d957b9876e03dd9ccbbd747e9795', 'new_sha256': '70ecbf0b02f3c17887538cdea79891d560a19dd16ca55cc79c8c9bd0deca406e'}, 'tests/test_phase4_citation_split.py': {'old_sha256': None, 'new_sha256': 'a2d931fba7d6c9122feaa5d1703372ebbbc0a66e35267135be6d6bbb450831c2'}}, 'api_calls': 0}
PATCH_FILES = {'rfp_phase4/citation_recovery.py': '"""Recover source quotations, then run the original strict validator.\n\nEvery non-whitespace character, the cited chunk, document, SHA and raw offsets\nremain authoritative. This never paraphrases, changes digits/punctuation,\nsearches other documents or invokes a model. Ambiguous recovery is rejected.\nOne omitted block between two uniquely matched whole-line source fragments can\nbe represented as TWO separate citations; a false continuous quote is never kept.\n"""\nfrom __future__ import annotations\n\nimport copy\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sqlite3\n\n\nRECOVERY_POLICY = \'unique_whitespace_or_two_whole_line_fragments_v2\'\nMIN_SPLIT_FRAGMENT_CHARS = 12\n\n\ndef _compact_with_positions(text):\n    positions = [i for i, char in enumerate(text) if not char.isspace()]\n    return \'\'.join(text[i] for i in positions), positions\n\n\ndef _recover_whitespace(source, quote):\n    needle, _ = _compact_with_positions(quote)\n    haystack, positions = _compact_with_positions(source)\n    if not needle:\n        return None, \'blank_quote\'\n    start = haystack.find(needle)\n    if start < 0:\n        return None, \'non_whitespace_characters_differ\'\n    if haystack.find(needle, start + 1) >= 0:\n        return None, \'ambiguous_whitespace_match\'\n    left, right = positions[start], positions[start + len(needle) - 1] + 1\n    restored = source[left:right]\n    assert \'\'.join(restored.split()) == \'\'.join(quote.split())\n    return {\'quote\': restored, \'quote_start\': left, \'quote_end\': right}, \'whitespace_only\'\n\n\ndef _source_lines(text):\n    """Nonempty full lines with exact original character offsets."""\n    result, offset = [], 0\n    for raw in text.splitlines(keepends=True):\n        positions = [i for i, char in enumerate(raw) if not char.isspace()]\n        if positions:\n            result.append({\'key\': \'\'.join(raw[i] for i in positions),\n                           \'start\': offset + positions[0],\n                           \'end\': offset + positions[-1] + 1})\n        offset += len(raw)\n    return result\n\n\ndef _recover_two_fragments(source, quote):\n    """Unique ordered full-line blocks, exactly one nonblank omitted source gap.\n\n    No character-level subsequence matching, intra-line omission, new words,\n    cross-chunk search, reordered blocks or short isolated labels are accepted.\n    """\n    requested = [line[\'key\'] for line in _source_lines(quote)]\n    lines = _source_lines(source)\n    if len(requested) < 2:\n        return None, \'not_a_multiline_whole_line_quote\'\n    keys = [line[\'key\'] for line in lines]\n    candidates = {}\n    for split in range(1, len(requested)):\n        first, second = requested[:split], requested[split:]\n        if min(len(\'\'.join(first)), len(\'\'.join(second))) < MIN_SPLIT_FRAGMENT_CHARS:\n            continue\n        starts_a = [i for i in range(len(keys) - len(first) + 1) if keys[i:i+len(first)] == first]\n        starts_b = [i for i in range(len(keys) - len(second) + 1) if keys[i:i+len(second)] == second]\n        for a in starts_a:\n            for b in starts_b:\n                if b <= a + len(first):  # Require at least one nonempty omitted source line.\n                    continue\n                x, y = lines[a][\'start\'], lines[a+len(first)-1][\'end\']\n                z, w = lines[b][\'start\'], lines[b+len(second)-1][\'end\']\n                identity = x, y, z, w\n                candidates[identity] = [\n                    {\'quote\': source[x:y], \'quote_start\': x, \'quote_end\': y},\n                    {\'quote\': source[z:w], \'quote_start\': z, \'quote_end\': w}]\n                if len(candidates) > 1:\n                    return None, \'ambiguous_fragment_mapping\'\n    if not candidates:\n        return None, \'no_unique_two_fragment_match\'\n    fragments = next(iter(candidates.values()))\n    assert \'\'.join(\'\'.join(f[\'quote\'].split()) for f in fragments) == \'\'.join(quote.split())\n    return fragments, \'noncontiguous_quote_split\'\n\n\ndef validate_with_quote_recovery(registry, payload, context_chunks, allowed_doc_ids=None):\n    """A formatting recovery layer; EvidenceRegistry itself remains strict.\n\n    Diagnostic quotations are unvalidated model data, never normal answer text.\n    Cache payloads and caller objects are not mutated. A single unresolved error\n    still prevents the whole model answer from being rendered.\n    """\n    strict = registry.validate_answer(payload, context_chunks, allowed_doc_ids=allowed_doc_ids)\n    report = {\'policy\': RECOVERY_POLICY, \'corrected_citations\': 0, \'split_citations\': 0,\n              \'additional_api_calls\': 0, \'strict_validation_initially_passed\': strict[\'valid\']}\n    strict[\'citation_recovery\'] = report\n    strict[\'citation_diagnostics\'] = []\n    if strict[\'valid\']:\n        return strict\n    try:\n        canonical = registry.validate_chunks(context_chunks, allowed_doc_ids=allowed_doc_ids)\n    except (TypeError, ValueError, KeyError):\n        return strict\n    if not isinstance(payload, dict) or not isinstance(payload.get(\'claims\'), list):\n        return strict\n    repaired = copy.deepcopy(payload)\n    context = {chunk[\'chunk_id\']: chunk for chunk in canonical}\n    diagnostics = []\n    corrections, splits = 0, 0\n    for claim_index, claim in enumerate(repaired[\'claims\'], 1):\n        if not isinstance(claim, dict) or not isinstance(claim.get(\'citations\'), list):\n            continue\n        replacement_citations = []\n        for citation_index, citation in enumerate(claim[\'citations\'], 1):\n            if not isinstance(citation, dict):\n                replacement_citations.append(citation)\n                continue\n            replacement_citations.append(citation)\n            cid, quote = citation.get(\'chunk_id\'), citation.get(\'quote\')\n            item = {\'claim_index\': claim_index, \'citation_index\': citation_index,\n                    \'claim_doc_id\': claim.get(\'doc_id\'), \'chunk_id\': cid,\n                    \'model_quote\': quote, \'quote_is_model_data_not_a_verified_claim\': True}\n            chunk = context.get(cid) if isinstance(cid, str) else None\n            if chunk is None:\n                item[\'outcome\'] = \'chunk_not_in_supplied_context\'\n            elif claim.get(\'doc_id\') != chunk[\'doc_id\']:\n                item[\'outcome\'] = \'claim_and_chunk_document_differ\'\n            elif not isinstance(quote, str) or not quote.strip():\n                item[\'outcome\'] = \'invalid_quote\'\n            elif quote in chunk[\'text\']:\n                item[\'outcome\'] = \'already_exact\'\n            else:\n                recovered, outcome = _recover_whitespace(chunk[\'text\'], quote)\n                item[\'outcome\'] = outcome\n                if recovered is not None:\n                    # Replacement is a literal source substring, not a normalized citation.\n                    citation[\'quote\'] = recovered[\'quote\']\n                    item[\'restored_source_quote\'] = recovered[\'quote\']\n                    item[\'source_chunk_start\'] = recovered[\'quote_start\']\n                    item[\'source_chunk_end\'] = recovered[\'quote_end\']\n                    corrections += 1\n                else:\n                    item[\'cited_chunk_preview\'] = chunk[\'text\'][:300]\n                    if outcome == \'non_whitespace_characters_differ\' and set(citation) == {\'chunk_id\',\'quote\'}:\n                        fragments, split_outcome = _recover_two_fragments(chunk[\'text\'], quote)\n                        item[\'split_check\'] = split_outcome\n                        if fragments is not None:\n                            replacement_citations.pop()\n                            replacement_citations.extend({\'chunk_id\':cid,\'quote\':f[\'quote\']} for f in fragments)\n                            item[\'outcome\'] = split_outcome\n                            item[\'fragments\'] = fragments\n                            left, right = fragments[0][\'quote_end\'], fragments[1][\'quote_start\']\n                            item[\'omitted_source_gap\'] = {\'start\':left,\'end\':right,\'preview\':chunk[\'text\'][left:right][:300]}\n                            item[\'display_as_separate_quotes\'] = True\n                            corrections += 1\n                            splits += 1\n            diagnostics.append(item)\n        claim[\'citations\'] = replacement_citations\n    validated = registry.validate_answer(repaired, canonical, allowed_doc_ids=allowed_doc_ids)\n    validated[\'citation_recovery\'] = {**report, \'corrected_citations\': corrections, \'split_citations\':splits}\n    validated[\'citation_diagnostics\'] = diagnostics\n    return validated\n\n\ndef _cached_response(root, saved):\n    """Read only the matching cache row, with checksum and original input checks."""\n    fingerprint = (saved.get(\'generation\') or {}).get(\'request_fingerprint\')\n    if not isinstance(fingerprint, str) or len(fingerprint) != 64:\n        raise ValueError(\'저장 결과에 요청 지문이 없습니다.\')\n    cache_path = Path(root) / \'phase4_outputs/generation_cache/grounded_responses.sqlite3\'\n    if not cache_path.is_file():\n        raise FileNotFoundError(\'이 프로젝트의 기존 모델 응답 캐시가 없습니다. 추가 API 호출은 하지 않았습니다.\')\n    with sqlite3.connect(cache_path.resolve().as_uri() + \'?mode=ro\', uri=True) as connection:\n        row = connection.execute(\'SELECT request_json,response_json,checksum FROM responses WHERE fingerprint=?\',\n                                 (fingerprint,)).fetchone()\n    if row is None:\n        raise ValueError(\'이 결과와 일치하는 캐시 응답이 없습니다. 추가 API 호출은 하지 않았습니다.\')\n    if hashlib.sha256(row[0].encode()).hexdigest() != fingerprint:\n        raise ValueError(\'캐시 요청 지문이 다릅니다.\')\n    if hashlib.sha256((row[0] + \'\\n\' + row[1]).encode()).hexdigest() != row[2]:\n        raise ValueError(\'캐시 응답 무결성 검사에 실패했습니다.\')\n    request, response = json.loads(row[0]), json.loads(row[1])\n    if response.get(\'request_fingerprint\') != fingerprint:\n        raise ValueError(\'캐시 응답이 다른 요청을 가리킵니다.\')\n    mode = response.get(\'execution_mode\')\n    if mode not in {\'live_openai\',\'injected_test_client\'} or request.get(\'execution_namespace\') != mode:\n        raise ValueError(\'캐시 요청과 응답의 실제/모의 실행 구분이 다릅니다.\')\n    user = json.loads(request[\'request\'][\'input\'][1][\'content\'])\n    expected_task = \'comparison\' if saved.get(\'retrieval\', {}).get(\'comparison\') else \'qa\'\n    if user.get(\'task\') != expected_task or user.get(\'question\') != saved.get(\'retrieval\', {}).get(\'standalone_query\'):\n        raise ValueError(\'캐시의 작업 또는 질문이 저장 결과와 다릅니다.\')\n    evidence = user.get(\'evidence\', [])\n    chunks = saved.get(\'source_chunks\', [])\n    if len(evidence) != len(chunks):\n        raise ValueError(\'캐시와 저장 결과의 입력 근거 개수가 다릅니다.\')\n    for original, chunk in zip(evidence, chunks):\n        for key in [\'chunk_id\',\'doc_id\',\'sha256\',\'text\']:\n            if original.get(key) != chunk.get(key):\n                raise ValueError(\'캐시와 저장 결과의 입력 근거가 다릅니다.\')\n    generation = saved.get(\'generation\') or {}\n    if response.get(\'execution_mode\') != generation.get(\'execution_mode\') or response.get(\'live_api\') != generation.get(\'live_api\'):\n        raise ValueError(\'캐시 응답의 모의/실제 실행 구분이 다릅니다.\')\n    return response\n\n\ndef replay_saved_result(root, result_path, *, registry=None, persist=True):\n    """Revalidate an existing response without API key, SDK or network calls."""\n    root, path = Path(root).resolve(), Path(result_path).resolve()\n    saved = json.loads(path.read_text(encoding=\'utf-8\'))\n    response = _cached_response(root, saved)\n    bootstrap_index = None\n    if registry is None:\n        from .runtime import bootstrap\n        app = bootstrap(root)\n        registry, bootstrap_index = app[\'registry\'], app[\'index\']\n    try:\n        checked = validate_with_quote_recovery(registry, response[\'payload\'], saved[\'source_chunks\'],\n                                               allowed_doc_ids=saved.get(\'active_doc_ids\') or None)\n        result = copy.deepcopy(saved)\n        result[\'validation\'] = checked\n        result[\'claims\'] = checked[\'claims\']\n        result[\'missing_information\'] = checked[\'missing_information\']\n        result[\'status\'] = checked[\'status\'] if checked[\'valid\'] else \'not_generated\'\n        result[\'execution_status\'] = \'completed\' if checked[\'valid\'] else \'validation_failed\'\n        result[\'answer_generated\'] = bool(checked[\'valid\'] and checked[\'claims\'])\n        result[\'markdown\'] = registry.render(checked)\n        if checked[\'valid\'] and saved.get(\'retrieval\', {}).get(\'comparison\'):\n            from .rag import _comparison_markdown\n            scopes = saved.get(\'active_doc_ids\') or []\n            cited = {claim[\'doc_id\'] for claim in checked[\'claims\']}\n            missing = [doc_id for doc_id in scopes if doc_id not in cited]\n            if missing:\n                checked[\'status\'] = \'partial\' if checked[\'claims\'] else \'insufficient_evidence\'\n                checked[\'missing_information\'].extend(f\'선택 문서 {did}: 검증된 비교 근거 미확보\' for did in missing)\n                result[\'status\'], result[\'missing_information\'] = checked[\'status\'],checked[\'missing_information\']\n            catalog = [{\'doc_id\': did, \'file_name\': registry.documents_by_id[did].get(\'file_name\',did)}\n                       for did in scopes]\n            result[\'markdown\'] = _comparison_markdown(checked, scopes, catalog)\n        result[\'replay\'] = {\'additional_api_calls\': 0, \'original_result_path\': str(path),\n                            \'scope\': \'Offline source revalidation of the original cached model response; no new semantic quality measurement.\'}\n        result[\'generation\'][\'original_usage\'] = response.get(\'original_usage\', response.get(\'usage\', {}))\n        result[\'generation\'][\'usage\'] = {\'api_calls\':0,\'input_tokens\':0,\'output_tokens\':0,\n                                          \'total_tokens\':0,\'cached_input_tokens\':0,\'usage_reported\':True}\n        result[\'generation\'][\'cache_hit\'] = True\n        result[\'generation\'][\'current_execution_mode\'] = \'offline_source_revalidation\'\n        result[\'generation\'][\'request_latency_ms\'] = 0.0\n        result[\'latency_ms\'] = None\n        if persist:\n            stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n            destination = path.with_name(path.stem + \'_revalidated_\' + stamp + \'.json\')\n            with destination.open(\'x\',encoding=\'utf-8\') as handle:\n                json.dump(result,handle,ensure_ascii=False,indent=2,allow_nan=False)\n                handle.write(\'\\n\')\n            result[\'revalidated_result_path\'] = str(destination)\n        return result\n    finally:\n        if bootstrap_index is not None:\n            bootstrap_index.close()\n', 'tests/test_phase4_citation_split.py': '"""A joined noncontiguous quote becomes two exact quotes, never a fake substring."""\nimport copy\nimport unittest\n\nfrom test_phase4_evidence import block,document,chunk,answer\nfrom rfp_phase4.evidence import EvidenceRegistry\nfrom rfp_phase4.citation_recovery import validate_with_quote_recovery\n\n\nHEADER=\'<부가세 포함, 금액단위 : 원>\'\nBODY=\'구       분\\n금      액\\n비     고\\n용 역 비\\n부가가치세 포함\\n장  비\\n구입비\\nHW 구매 부문\\n부가가치세 포함\'\nGAP=\'발주자\\n한국주택금융공사\\n직전도급자\\n-\\n하도급\\n주는자\\n사업명\\n계약금액\\n상  호\\n대표자\'\n\n\nclass CitationSplitTests(unittest.TestCase):\n    def check(self,source,quote,*,did=\'d1\',allowed=None):\n        b=block(text=source);d=document(source=b);c=chunk(b)\n        reg=EvidenceRegistry([d],[c]);payload=answer(quote=quote,did=did)\n        original=copy.deepcopy(payload)\n        before=reg.validate_answer(payload,[c],allowed)\n        after=validate_with_quote_recovery(reg,payload,[c],allowed)\n        self.assertEqual(payload,original)\n        return before,after,d,c\n\n    def test_reported_header_plus_table_body_is_two_exact_citations(self):\n        before,r,d,c=self.check(HEADER+\'\\n\'+GAP+\'\\n\'+BODY+\'\\nSW 구매 부문\\n부가가치세 포함\',HEADER+\'\\n\'+BODY)\n        self.assertFalse(before[\'valid\']);self.assertTrue(r[\'valid\'])\n        citations=r[\'claims\'][0][\'citations\']\n        self.assertEqual(len(citations),2)\n        self.assertEqual([v[\'quote\'] for v in citations],[HEADER,BODY])\n        self.assertEqual(r[\'citation_recovery\'][\'split_citations\'],1)\n        self.assertEqual(r[\'citation_recovery\'][\'additional_api_calls\'],0)\n        for v in citations:\n            self.assertIn(v[\'quote\'],c[\'text\'])\n            for span in v[\'source_ranges\']:\n                self.assertEqual(d[\'blocks\'][0][\'text\'][span[\'char_start\']:span[\'char_end\']],\n                                 c[\'text\'][span[\'chunk_start\']:span[\'chunk_end\']])\n        self.assertNotIn(GAP,\' \'.join(v[\'quote\'] for v in citations))\n\n    def test_whitespace_inside_each_fragment_is_restored(self):\n        _,r,_,_=self.check(HEADER+\'\\n\'+GAP+\'\\n\'+BODY,HEADER.replace(\' \',\'\')+\'\\n\'+\'\\n\'.join(s.replace(\' \',\'\') for s in BODY.splitlines()))\n        self.assertTrue(r[\'valid\']);self.assertEqual(r[\'claims\'][0][\'citations\'][1][\'quote\'],BODY)\n\n    def test_changed_numbers_vat_units_punctuation_and_words_are_rejected(self):\n        source=HEADER+\'\\n\'+GAP+\'\\n사업예산은 688백만원이며 부가세 포함입니다.\'\n        for tail in [\'사업예산은 699백만원이며 부가세 포함입니다.\',\'사업예산은 688백만원이며 부가세 별도입니다.\',\n                     \'사업예산은 688달러이며 부가세 포함입니다.\',\'사업예산은 688백만원이며 부가세 포함입니다!\',\n                     \'사업예산은 688백만원으로 확정되었습니다.\']:\n            self.assertFalse(self.check(source,HEADER+\'\\n\'+tail)[1][\'valid\'])\n\n    def test_intra_line_omission_and_negation_removal_are_rejected(self):\n        source=HEADER+\'\\n\'+GAP+\'\\n사업예산은 688백만원이며 부가세 포함이 아닙니다.\'\n        r=self.check(source,HEADER+\'\\n사업예산은 688백만원이며 부가세 포함입니다.\')[1]\n        self.assertFalse(r[\'valid\'])\n\n    def test_reordered_fragments_are_rejected(self):\n        self.assertFalse(self.check(HEADER+\'\\n\'+GAP+\'\\n\'+BODY,BODY+\'\\n\'+HEADER)[1][\'valid\'])\n\n    def test_duplicate_fragment_placements_are_ambiguous(self):\n        source=HEADER+\'\\n\'+GAP+\'\\n\'+BODY+\'\\n여백\\n\'+BODY\n        r=self.check(source,HEADER+\'\\n\'+BODY)[1]\n        self.assertFalse(r[\'valid\'])\n        self.assertEqual(r[\'citation_diagnostics\'][0][\'split_check\'],\'ambiguous_fragment_mapping\')\n\n    def test_more_than_one_omitted_block_is_not_accepted(self):\n        a=\'사업예산은 688백만원입니다.\';b=\'사업기간은 계약일부터 5개월입니다.\';c=\'안정화 기간은 1개월 포함입니다.\'\n        source=a+\'\\n생략 구간 하나\\n\'+b+\'\\n생략 구간 둘\\n\'+c\n        self.assertFalse(self.check(source,a+\'\\n\'+b+\'\\n\'+c)[1][\'valid\'])\n\n    def test_short_isolated_labels_do_not_enable_split(self):\n        source=\'예산\\n다른 문장\\n부가세는 포함이며 예산은 688백만원입니다.\'\n        self.assertFalse(self.check(source,\'예산\\n부가세는 포함이며 예산은 688백만원입니다.\')[1][\'valid\'])\n\n    def test_scope_and_document_checks_still_apply(self):\n        source=HEADER+\'\\n\'+GAP+\'\\n\'+BODY;quote=HEADER+\'\\n\'+BODY\n        self.assertFalse(self.check(source,quote,allowed=[])[1][\'valid\'])\n        self.assertFalse(self.check(source,quote,did=\'other\')[1][\'valid\'])\n\n    def test_true_contiguous_quote_is_not_split(self):\n        _,r,_,_=self.check(HEADER+\'\\n\'+BODY,HEADER+\'\\n\'+BODY)\n        self.assertTrue(r[\'valid\']);self.assertEqual(r[\'citation_recovery\'][\'split_citations\'],0)\n        self.assertEqual(len(r[\'claims\'][0][\'citations\']),1)\n\n    def test_replaying_revalidated_output_keeps_original_usage(self):\n        import test_phase4_citation_recovery as fixtures\n        from rfp_phase4.citation_recovery import replay_saved_result\n        fixture=fixtures.ReplayTests();fixture.setUp();self.addCleanup(fixture.doCleanups)\n        first=replay_saved_result(fixture.root,fixture.path,registry=fixture.reg)\n        second=replay_saved_result(fixture.root,first[\'revalidated_result_path\'],registry=fixture.reg)\n        self.assertEqual(second[\'generation\'][\'original_usage\'][\'api_calls\'],1)\n        self.assertEqual(second[\'generation\'][\'usage\'][\'api_calls\'],0)\n\n\nif __name__==\'__main__\':unittest.main()\n'}
space = {'__name__':'citation_split_installer','__file__':str(ROOT / '_embedded_split_installer.py')}
exec(compile(INSTALLER_SOURCE, '<reviewed split installer>', 'exec'), space)
patch_report = space['apply_patch'](ROOT, manifest=PATCH_MANIFEST, embedded_files=PATCH_FILES)
print(json.dumps(patch_report, ensure_ascii=False, indent=2))


{
  "patch_id": "phase4-citation-split-fix-v2",
  "modified_files": [
    "rfp_phase4/citation_recovery.py",
    "tests/test_phase4_citation_split.py",
    "notebooks/05_younggwan_phase4_grounded_answers.ipynb"
  ],
  "already_applied_files": [],
  "backup_directory": "/Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/patch_backups/20261111T094102_009915Z",
  "api_calls": 0,
  "data_or_index_modified": false,
  "cache_modified": false,
  "restart_existing_phase4_notebook_kernel": true
}


## 2. 같은 캐시 응답 다시 검사

새 모델 요청 없이 같은 요청 지문·캐시 체크섬·질문·원문 근거를 확인하고, 분리된 인용도 기존 엄격한 출처 검사를 통과하는지 확인합니다.
원래 파일 옆에 새 `_revalidated_...json`을 저장합니다. 캐시가 없으면 추가 호출을 하지 않고 멈춥니다.


In [3]:
import importlib
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
recovery = importlib.import_module('rfp_phase4.citation_recovery')
if Path(recovery.__file__).resolve().parents[1] != ROOT:
    raise RuntimeError('다른 프로젝트가 로딩되어 있습니다. 커널을 재시작하고 다시 실행하세요.')
recovery = importlib.reload(recovery)
replayed = recovery.replay_saved_result(ROOT, ROOT / FAILED_RESULT)
print(replayed['markdown'])
print('상태:', replayed['status'], '/', replayed['execution_status'])
details = replayed['validation'].get('citation_recovery', {})
print('복원·분리한 인용 수:', details.get('corrected_citations', 0))
print('떨어진 원문 구간을 별도 인용으로 분리한 수:', details.get('split_citations', 0))
print('추가 API 호출:', replayed['replay']['additional_api_calls'])
if not replayed['validation']['valid']:
    print('실패 사유:', replayed['validation']['errors'])
    print('아래는 검증에 실패한 인용 데이터이며 정상 답변이 아닙니다.')
    bad = [d for d in replayed['validation'].get('citation_diagnostics', [])
           if d.get('outcome') not in {'already_exact','whitespace_only','noncontiguous_quote_split'}]
    print(json.dumps(bad, ensure_ascii=False, indent=2))
print('새 결과:', replayed.get('revalidated_result_path'))


| 항목 | 한국주택금융공사\_2023\_HF 인사이트 시스템 개발\_제안요청서.hwp | 한국주택금융공사\_2021\_부동산 등기정보 시스템 구축\_제안요청서.hwp |
| --- | --- | --- |
| 예산 | HF 인사이트 시스템 개발의 사업예산은 688백만원 (부가세포함)이며, 개발예산 498백만원과 HW구입 190백만원으로 구성된다. [1][2][3] | 부동산 등기정보 시스템 구축의 사업예산은 280백만 원 이내(부가세 포함)이다. [5] |
| 사업기간 | HF 인사이트 시스템 개발의 사업기간은 계약일로부터 5개월이며, 안정화·시스템모니터링 1개월이 포함된다. [4] | 부동산 등기정보 시스템 구축의 사업기간은 계약체결일로부터 365일이며, 외부 정보연계 등과 관련한 협의 지연 시 단계적 오픈 및 과업 조정 가능하다. [6] |

원문 근거:

1. 한국주택금융공사\_2023\_HF 인사이트 시스템 개발\_제안요청서.hwp · BodyText/Section0 / paragraph 93 … BodyText/Section0 / paragraph 131 — “□ 사업예산 : 688백만원 (부가세포함) / ㅇ 개발예산 (498백만원) 및  HW구입(190백만원)”
2. 한국주택금융공사\_2023\_HF 인사이트 시스템 개발\_제안요청서.hwp · BodyText/Section0 / paragraph 4081 … BodyText/Section0 / paragraph 4172 — “&lt;부가세 포함, 금액단위 : 원&gt;”
3. 한국주택금융공사\_2023\_HF 인사이트 시스템 개발\_제안요청서.hwp · BodyText/Section0 / paragraph 4081 … BodyText/Section0 / paragraph 4172 — “구       분 / 금      액 / 비     고 / 용 역 비 / 부가가치세 포함 / 장  비 / 구입비 / HW 구매 부문 / 부가가치세 포함”
4. 한국주택금융공사\_2023\_HF 인사이트 시스템 개발\

## 3. 기존 실습 계속하기

상태가 `completed`이고 분리한 인용 수가 1이면 이번에 제보한 인용 연결 문제가 처리된 것입니다.
이후 기존 05 노트북의 커널을 재시작하고 비교 셀까지 실행하세요. 모델 캐시는 유지되어 같은 요청에 재사용할 수 있습니다.
실제로 추가 호출했는지는 사용량과 캐시 적중 기록으로 확인하세요.

계속 실패하면 새 `_revalidated_...json`의 진단을 확인합니다. 답변의 의미·금액·VAT·기간 조건은 표시된 원문 인용과 별도로 대조해야 합니다.
